# Professional Multi-Label Classification Pipeline with ResNet50

This Jupyter notebook demonstrates how to utilize the modular architecture defined in the `src/` directory to train and run inference on the satellite image dataset of the Amazon basin.

### Advantages of this structure:
1. **Modularity:** Centralized code (Dataset, Model, Training loops, and Utilities) resides in dedicated `.py` files.
2. **Dynamic Configuration:** Paths adapt automatically whether running on Kaggle or locally via `src/config.py`.
3. **Readability:** This notebook focuses exclusively on the execution flow, metrics visualization, and dynamic inference.

### 1. Initialization and Data Loading

In [ ]:
import os
import pandas as pd
from src.config import CSV_PATH, IMG_PATH_TRAIN, TEST_IMG_PATH
from src.utils import visualizar_imagen

# To load dataset labels from the CSV file
df = pd.read_csv(CSV_PATH)
print(f"Total training images: {len(df)}")
print("\nFirst rows of the CSV:")
print(df.head())

# To display a sample from the training dataset
sample_row = df.iloc[4]
img_name = sample_row['image_name'] + '.jpg'
img_tags = sample_row['tags']

image_path = os.path.join(IMG_PATH_TRAIN, img_name)
visualizar_imagen(image_path, title=f"Sample: {img_name}\nLabels: {img_tags}")

### 2. Label Representation (One-Hot Encoding)
Given that this is a **multi-label classification** problem (where each image can have multiple labels simultaneously, e.g., `clear`, `primary`, `water`, etc.), the conventional single-class output approach is not applicable.

Instead, **One-Hot Encoding** is used to represent the target as a binary vector of size $17$ (the total number of unique classes). In this vector, each position represents a specific category and takes the value $1$ if the category is present in the image, and $0$ otherwise.

Internal functions in `src/dataset.py` build the mapping dictionaries between words and indices, and seamlessly encode these labels into tensors during data loading.

### 3. Dataset and DataLoader Preparation

In [ ]:
import torch
from torch.utils.data import DataLoader
from src.dataset import AmazonDataset, get_transforms, get_unique_tags
from src.config import BATCH_SIZE

# To map labels internally
etiquetas_unicas, tag_to_idx, idx_to_tag = get_unique_tags(df)
num_clases = len(etiquetas_unicas)

# To obtain standard ResNet50 transformations
transformaciones = get_transforms()

# To instantiate the dataset for dimension verification
dataset_completo = AmazonDataset(
    df=df,
    img_dir=IMG_PATH_TRAIN,
    tag_to_idx=tag_to_idx,
    transform=transformaciones
)

train_loader = DataLoader(dataset_completo, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

# To verify batch dimensions
imagenes_lote, targets_lote = next(iter(train_loader))
print(f"Image batch shape: {imagenes_lote.shape}")   # Expected: [32, 3, 224, 224]
print(f"Target batch shape: {targets_lote.shape}")  # Expected: [32, 17]

### 4. Model Definition (AmazonResnet)
The architecture is based on a **ResNet50** pre-trained on ImageNet. To adapt it to our 17-class multi-label classification problem, the final fully connected layer (`fc`) is replaced.

In [ ]:
from src.model import AmazonResnet

# To create the model adapted for 17 classes with a frozen backbone
modelo = AmazonResnet(num_classes=num_clases, freeze_backbone=True)
print(modelo)

# To perform a dummy forward pass to verify dimensions
lote_ficticio = torch.randn(4, 3, 224, 224)
with torch.no_grad():
    salida_ficticia = modelo(lote_ficticio)
print(f"\nOutput shape: {salida_ficticia.shape}")  # Expected: [4, 17]

### 5. Training and Local Validation (F1-Score)
To validate the network properly without overfitting, the dataset is split into $80\%$ training and $20\%$ validation. The loss is optimized using **BCEWithLogitsLoss** and the **Adam** optimizer.

In [ ]:
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from src.train import entrenamiento_validacion
from src.config import LEARNING_RATE

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Running on device: {device}')

# To split the dataset into training and validation sets
df_train, df_val = train_test_split(df, test_size=0.2, random_state=42)
print(f"Training samples: {len(df_train)}")
print(f"Validation samples: {len(df_val)}")

# To instantiate training and validation datasets
dataset_train = AmazonDataset(df=df_train, img_dir=IMG_PATH_TRAIN, tag_to_idx=tag_to_idx, transform=transformaciones)
dataset_val = AmazonDataset(df=df_val, img_dir=IMG_PATH_TRAIN, tag_to_idx=tag_to_idx, transform=transformaciones)

train_loader_split = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader_split = DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# To configure the model and optimizer
modelo_val = AmazonResnet(num_classes=num_clases, freeze_backbone=True).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer_val = optim.Adam([p for p in modelo_val.parameters() if p.requires_grad], lr=LEARNING_RATE)

# To run the local validation loop for 2 epochs, uncomment the following line:
# entrenamiento_validacion(modelo_val, train_loader_split, val_loader_split, criterion, optimizer_val, device, epochs=2)

### 6. Loading Pre-trained Weights and Running Inference
To avoid re-training the neural network (a process requiring high computational power and a GPU), it is possible to directly load the weights file obtained from a previous cloud training: **`amazon_resnet50_v1.pth`**.

With these weights loaded, the model achieves a **88.66%** F1-Score on the local validation set and a validation loss of **0.1045**, producing excellent predictions.

In [ ]:
from src.utils import ejecutar_inferencia_test

PESOS_PATH = "amazon_resnet50_v1.pth"

if os.path.exists(PESOS_PATH):
    print(f"Loading pre-trained weights from {PESOS_PATH}...")
    modelo.load_state_dict(torch.load(PESOS_PATH, map_location=device))
    modelo.to(device)
    print("Weights loaded successfully!")
else:
    print(f"Warning: Weights file not found at '{PESOS_PATH}'.")

# To run inference on the test set if available
if os.path.exists(TEST_IMG_PATH):
    archivos_test = os.listdir(TEST_IMG_PATH)
    print(f"Total test images detected: {len(archivos_test)}")
    
    # To run inference in the range [9:12]
    ejecutar_inferencia_test(
        modelo=modelo,
        test_img_path=TEST_IMG_PATH,
        archivos_test=archivos_test,
        transformaciones=transformaciones,
        idx_to_tag=idx_to_tag,
        device=device,
        start_idx=9,
        end_idx=12,
        umbral=0.5
    )
else:
    print(f"Test image directory not found at: {TEST_IMG_PATH}")